# Exploring the Virchow embeddings

Runs on a laptop CPU, in a few minutes. Unzip `pilot_outputs.zip` from Kaggle into the repo folder, so that the slide files sit in `features/` at the top of the repo.

Each slide is one `.pt` file holding its tiles' Virchow embeddings (2,560 numbers per tile). A slide's embedding is the mean of its tiles, normalised to length 1.

Nothing here is trained on the diagnoses. These are checks of what Virchow already captures, before any classifier.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "..")  # the notebook runs from notebooks/, the code is one level up
import numpy as np, pandas as pd
from brainpath import explore

FEATURES = "../features"  # folder with the .pt files, at the top of the repo
slides = explore.load(FEATURES)
table = explore.slide_table(slides)
X = explore.slide_matrix(slides)
print(len(slides), "slides,", table.tumour_type.nunique(), "types, embedding size", X.shape[1])

## 1. Inventory

Tiles per slide. Small slides (a few hundred tiles) give noisier embeddings, so keep them in mind when one looks odd later.

In [ ]:
display(table.groupby(["role", "tumour_type"], sort=False).n_tiles.describe()[["count", "min", "50%", "max"]])
table.sort_values("n_tiles").head(8)

## 2. PCA: how many directions carry the variance?

If a handful of components carry most of the variance, the slides differ along a few main axes, which may be tumour type or may be something technical (step 6 checks which).

In [ ]:
fig, cum = explore.plot_explained_variance(X)

## 3. Maps of the slides

Each point is a slide. Colour shows its role and marker shape its type, and each type's name sits at the centre of its slides. Linear PCA first, then UMAP (or t-SNE if UMAP is not installed), which shows local neighbourhoods better but distorts distances.

Look for: do slides of one type sit together? Do the held-out (green) types land in their own place, or on top of a known type?

In [ ]:
Y, name = explore.embed_2d(X, "pca")
explore.plot_map(Y, table, name);

In [ ]:
Y, name = explore.embed_2d(X, "umap")
explore.plot_map(Y, table, name);

## 4. Similarity between slides

Cosine similarity for every pair of slides, ordered by type. Dark squares on the diagonal mean slides of a type look alike. Dark blocks off the diagonal show which types Virchow confuses.

In [ ]:
explore.plot_similarity(X, table);

## 5. Separability

- **silhouette** (−1 to 1): above 0 means a slide is closer to its own type than to the others.
- **nn_accuracy**: leave-one-out nearest-neighbour accuracy. It is the simplest possible classifier, with no training.
- **nearest_known_sim**: similarity to the closest slide of a known type. If held-out slides score clearly lower than known ones, unseen types should be detectable.

In [ ]:
sep, per_type, summary = explore.separability(X, table)
display(per_type.round(2))
summary

In [ ]:
explore.plot_heldout_distance(sep);

## 6. Confounders

Spearman correlation of the first principal components with traits that are not tumour biology. A strong correlation (|ρ| above about 0.5) with tile count or tissue area means that direction partly measures slide size, not tissue type.

In [ ]:
conf = explore.confounders(X, table)
conf.pivot(index="trait", columns="pc", values="spearman").round(2)

## 7. Tile clusters

k-means on tiles from all slides (300 per slide). Each cluster is a recurring tissue pattern. The table shows what share of each type's tiles falls in each cluster. Types that share clusters share tissue patterns, for example necrosis or normal brain.

To see what a cluster looks like, you would need the tile images, which were deleted after extraction. The tile coordinates are saved, so a few slides can be re-downloaded to view them.

In [ ]:
share = explore.tile_clusters(slides, k=8)
explore.plot_cluster_share(share);

## 8. Inside one slide

Each tile is coloured by its first three principal components (red, green and blue). Regions of similar colour are similar tissue. Compare it with the slide's thumbnail on EBRAINS if you open it there.

In [ ]:
for t, grp in table.groupby("tumour_type", sort=False):
    explore.plot_tile_map(slides[grp.index[grp.n_tiles.argmax()]])

## Notes for the write-up

Write down what you see before training: which types cluster, which get confused, whether held-out types stand apart, and any confounders. These observations motivate the method choices in the paper.